# Conformal Prediction Tutorial

## Introduction

**Conformal prediction** is a framework for constructing prediction intervals (regression) or prediction sets (classification) with **finite-sample, distribution-free coverage guarantees**.

Unlike Bayesian methods (Laplace, VI) or ensembles that produce approximate uncertainty estimates requiring calibration assumptions, conformal prediction guarantees:

$$P(Y_{n+1} \in C(X_{n+1})) \geq 1 - \alpha$$

for any user-chosen miscoverage level $\alpha$, under the sole assumption of **exchangeability** of the data.

### Key advantages:
- **No distributional assumptions** on the data or model
- **Works with any black-box model** (neural networks, random forests, etc.)
- **Exact finite-sample coverage** (not asymptotic)
- **Simple to implement** — just requires a held-out calibration set

### Methods covered in this tutorial:
1. **Split Conformal Regression** — constant-width intervals
2. **Conformalized Quantile Regression (CQR)** — adaptive intervals
3. **Conformal Classification (APS/RAPS)** — prediction sets
4. **Conformalized Laplace** — calibrated Bayesian intervals
5. **Conformalized Deep Ensemble** — calibrated ensemble intervals

## Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

from deepuq.models import MLP
from deepuq.methods import (
    SplitConformalRegressor,
    ConformalClassifier,
    CQRPredictor,
    ConformalUQWrapper,
    LaplaceWrapper,
    DeepEnsembleRegressor,
)

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 11

In [ ]:
# Generate synthetic sine regression data with heteroscedastic noise
def generate_sine_data(n=1000):
    x = torch.linspace(-3, 3, n).unsqueeze(1)
    noise_std = 0.1 + 0.3 * torch.abs(x)  # heteroscedastic noise
    y = torch.sin(x) + noise_std * torch.randn_like(x)
    return x, y.squeeze(1), noise_std.squeeze(1)

x_all, y_all, true_noise = generate_sine_data(1200)

# Train / Calibration / Test split
n_train, n_cal = 600, 300
x_train, y_train = x_all[:n_train], y_all[:n_train]
x_cal, y_cal = x_all[n_train:n_train+n_cal], y_all[n_train:n_train+n_cal]
x_test, y_test = x_all[n_train+n_cal:], y_all[n_train+n_cal:]

# Sort test set for clean plotting
sort_idx = x_test.squeeze().argsort()
x_test, y_test = x_test[sort_idx], y_test[sort_idx]

print(f"Train: {len(x_train)}, Cal: {len(x_cal)}, Test: {len(x_test)}")

## 1. Split Conformal Regression

The simplest conformal method: compute residuals on a calibration set, take the appropriate quantile, and form constant-width intervals around model predictions.

In [ ]:
# Train a simple MLP
model_reg = MLP(input_dim=1, hidden_dims=[64, 64], output_dim=1)

train_loader = DataLoader(TensorDataset(x_train, y_train.unsqueeze(1)), batch_size=64, shuffle=True)

optimizer = torch.optim.Adam(model_reg.parameters(), lr=1e-3)
for epoch in range(200):
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = nn.MSELoss()(model_reg(xb), yb)
        loss.backward()
        optimizer.step()

print(f"Training complete. Final loss: {loss.item():.4f}")

In [ ]:
# Calibrate split conformal
alpha = 0.1  # target miscoverage rate (90% coverage)
cal_data = TensorDataset(x_cal, y_cal)

model_reg.eval()
conformal_reg = SplitConformalRegressor(model_reg, alpha=alpha)
conformal_reg.calibrate(cal_data)

# Predict on test set
result = conformal_reg.predict_uq(x_test)
lower = result.metadata["conformal_lower"]
upper = result.metadata["conformal_upper"]

# Compute empirical coverage
covered = ((y_test >= lower) & (y_test <= upper)).float().mean()
print(f"Target coverage: {1-alpha:.0%}")
print(f"Empirical coverage: {covered:.1%}")
print(f"Interval width (constant): {(upper - lower).mean():.3f}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
x_np = x_test.squeeze().numpy()

ax.scatter(x_np, y_test.numpy(), s=5, alpha=0.4, label="Test data", color="gray")
ax.plot(x_np, result.mean.numpy(), "b-", lw=2, label="Prediction")
ax.fill_between(x_np, lower.numpy(), upper.numpy(), alpha=0.3, color="blue",
                label=f"90% Conformal interval (coverage={covered:.1%})")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Split Conformal Regression — Constant-Width Intervals")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Conformalized Quantile Regression (CQR)

CQR produces **adaptive** intervals that are wider in regions of high uncertainty. It requires a model that outputs lower and upper quantile predictions, which are then conformalized for exact coverage.

In [ ]:
# Train a quantile regression model (outputs 2 values: lower and upper quantiles)
model_qr = MLP(input_dim=1, hidden_dims=[64, 64], output_dim=2)

# Pinball loss for quantile regression
def pinball_loss(pred, target, quantiles):
    """Compute pinball loss for multiple quantiles."""
    target = target.unsqueeze(1).expand_as(pred)
    errors = target - pred
    loss = torch.zeros_like(pred)
    for i, q in enumerate(quantiles):
        loss[:, i] = torch.where(errors[:, i] >= 0, q * errors[:, i], (q - 1) * errors[:, i])
    return loss.mean()

quantiles = [alpha / 2, 1 - alpha / 2]  # 0.05 and 0.95

optimizer_qr = torch.optim.Adam(model_qr.parameters(), lr=1e-3)
for epoch in range(300):
    for xb, yb in train_loader:
        optimizer_qr.zero_grad()
        pred = model_qr(xb)
        loss = pinball_loss(pred, yb.squeeze(), quantiles)
        loss.backward()
        optimizer_qr.step()

print(f"Quantile regression training complete. Final loss: {loss.item():.4f}")

In [ ]:
# Calibrate CQR
model_qr.eval()
cqr = CQRPredictor(model_qr, alpha=alpha)
cqr.calibrate(cal_data)

# Predict
result_cqr = cqr.predict_uq(x_test)
lower_cqr = result_cqr.metadata["conformal_lower"]
upper_cqr = result_cqr.metadata["conformal_upper"]

# Coverage
covered_cqr = ((y_test >= lower_cqr) & (y_test <= upper_cqr)).float().mean()
print(f"CQR empirical coverage: {covered_cqr:.1%}")
print(f"CQR average interval width: {(upper_cqr - lower_cqr).mean():.3f}")
print(f"Split conformal average width: {(upper - lower).mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Split conformal
ax = axes[0]
ax.scatter(x_np, y_test.numpy(), s=5, alpha=0.4, color="gray")
ax.plot(x_np, result.mean.numpy(), "b-", lw=2)
ax.fill_between(x_np, lower.numpy(), upper.numpy(), alpha=0.3, color="blue")
ax.set_title(f"Split Conformal (constant width={upper[0].item()-lower[0].item():.2f})")
ax.set_xlabel("x"); ax.set_ylabel("y")

# CQR
ax = axes[1]
ax.scatter(x_np, y_test.numpy(), s=5, alpha=0.4, color="gray")
ax.plot(x_np, result_cqr.mean.numpy(), "r-", lw=2)
ax.fill_between(x_np, lower_cqr.numpy(), upper_cqr.numpy(), alpha=0.3, color="red")
ax.set_title(f"CQR (adaptive, avg width={(upper_cqr-lower_cqr).mean():.2f})")
ax.set_xlabel("x"); ax.set_ylabel("y")

plt.suptitle("Split Conformal vs CQR: Adaptive intervals capture heteroscedastic noise", y=1.02)
plt.tight_layout()
plt.show()

## 3. Conformal Classification (APS / RAPS)

For classification, conformal prediction produces **prediction sets** — subsets of classes that are guaranteed to contain the true class with probability at least $1 - \alpha$.

- **APS** (Adaptive Prediction Sets): includes classes in decreasing probability order until cumulative probability exceeds threshold
- **RAPS** (Regularized APS): penalizes large prediction sets to produce smaller, more informative sets